In [1]:
EXTRACT_USER_INFO_PROMPT = """
You are an expert user-information extraction, validation, and merging system.

Your task is to produce the FINAL UserInformation by checking:

1. Existing user information
2. The provided resume
3. Optional user instruction

The goal is to keep existing valid information and add/update information
when supported by the resume or user instruction.

============================================================
EXISTING USER INFORMATION
============================================================

{existing_user_info}


============================================================
RESUME
============================================================

{resume_content}


============================================================
OPTIONAL USER INSTRUCTION
============================================================

{user_instruction}


============================================================
MERGE RULES
============================================================

- Keep existing valid information.
- Check the resume for additional information that can be added.
- Check the user instruction for additional information or corrections.
- Add new valid information found in the resume or user instruction.
- If the user instruction explicitly asks to change/update information,
  follow the instruction when it provides a clear value.
- Do not invent, guess, or fabricate information.
- Do not remove valid existing information just because it is absent
  from the resume.
- Do not add duplicate values.
- For phone and email, keep all unique valid values as a
  comma-separated string.
- If no valid value exists for a field, return an empty string.


============================================================
FIELD VALIDATION
============================================================

Every extracted value must actually belong to its field.

full_name:
- Only the person's actual full name.
- Do not include job titles, company names, descriptions, emails,
  phone numbers, or URLs.

phone:
- Only phone numbers belonging to the person.
- Multiple phone numbers must be comma-separated.
- Do not include emails, URLs, or unrelated/company phone numbers.

email:
- Only email addresses belonging to the person.
- Multiple email addresses must be comma-separated.
- Do not include URLs or unrelated/company/recruiter emails unless
  clearly identified as the person's own email.

linkedin_url:
- Only the person's LinkedIn profile URL.
- It must actually be a LinkedIn URL.
- Do not put email, phone, GitHub, portfolio, or other URLs here.

github_url:
- Only the person's GitHub profile URL.
- It must actually be a GitHub URL.
- Do not put email, phone, LinkedIn, portfolio, or other URLs here.

portfolio_url:
- Only the person's personal portfolio/personal website URL.
- Do not put LinkedIn, GitHub, email, phone, company websites,
  or unrelated websites here.


============================================================
USER INSTRUCTION
============================================================

The user instruction is optional.

If it is empty, ignore it.

If provided, use it to determine whether information should be added
or updated.

The instruction may provide a new value, correct existing information,
or clarify which information from the resume belongs to the user.

Follow explicit user instructions when they provide a clear value.

However, still validate the value against the correct field type.

For example:

User instruction:
"Add my new email: anirban.work@gmail.com"

Then email should contain:
"anirban.work@gmail.com"

User instruction:
"Use this LinkedIn: https://linkedin.com/in/anirban"

Then linkedin_url should contain:
"https://linkedin.com/in/anirban"

Do not put the instruction itself into a field.


============================================================
VALUE FORMAT
============================================================

Every field must contain ONLY the actual value.

Never return an explanation, sentence, label, or description.

Examples:

Correct:
full_name = "Anirban Das"

Wrong:
full_name = "The user's full name is Anirban Das"

Wrong:
full_name = "Name: Anirban Das"


Correct:
email = "anirban@gmail.com, anirban.work@gmail.com"

Wrong:
email = "The user's emails are anirban@gmail.com, anirban.work@gmail.com"


Correct:
linkedin_url = "https://linkedin.com/in/anirban"

Wrong:
linkedin_url = "The user's LinkedIn is https://linkedin.com/in/anirban"


Do not include:
- field names inside values
- explanations
- labels such as "Name:", "Email:", "Phone:", etc.
- sentences
- comments
- prefixes or suffixes


============================================================
FINAL CHECK
============================================================

Before returning the result, verify every field:

1. Is the value actually for the user?
2. Does the value belong to the correct field?
3. Is the value supported by existing information, the resume,
   or the user instruction?
4. Is it free from duplicate values?
5. Does it contain ONLY the actual value?
6. For URLs, is it actually the correct type of URL?
7. For phone/email, are multiple values comma-separated?


============================================================
IMPORTANT
============================================================

Check the semantic meaning and format of every extracted value before
adding it.

For example:

- An email must be an email, not a URL.
- A LinkedIn URL must be a LinkedIn profile URL, not merely a URL
  containing "linkedin" somewhere in text.
- A GitHub URL must be a GitHub profile URL.
- A portfolio URL must be a personal website/portfolio, not LinkedIn
  or GitHub.
- A phone value must actually be a phone number.
- A person's name must actually represent the person.

Only merge information when it is reasonably clear that it belongs
to the user.

Return ONLY the structured UserInformation output.

{format_instructions}
"""

In [2]:
from pydantic import BaseModel, Field


class UserInformation(BaseModel):
    full_name: str = Field(
        default="",
        description="The full name of the person."
    )

    phone: str = Field(
        default="",
        description=(
            "All phone numbers belonging to the person, "
            "as a comma-separated string. Empty string if not found."
        )
    )

    linkedin_url: str = Field(
        default="",
        description="The person's LinkedIn profile URL. Empty string if not found."
    )

    github_url: str = Field(
        default="",
        description="The person's GitHub profile URL. Empty string if not found."
    )

    portfolio_url: str = Field(
        default="",
        description="The person's personal portfolio URL. Empty string if not found."
    )

    email: str = Field(
        default="",
        description=(
            "All email addresses belonging to the person, "
            "as a comma-separated string. Empty string if not found."
        )
    )

In [4]:
import json
import os

from dotenv import load_dotenv

from langchain_huggingface import (
    HuggingFaceEndpoint,
    ChatHuggingFace,
)

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_classic.output_parsers import OutputFixingParser

# from .extract_user_info_prompt import EXTRACT_USER_INFO_PROMPT
# from .extract_user_info_schema import UserInformation


load_dotenv()


llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    huggingfacehub_api_token=os.environ["HUGGINGFACEHUB_API_TOKEN1"],
    max_new_tokens=1024,
    temperature=0.1,
)

model = ChatHuggingFace(llm=llm)


parser = PydanticOutputParser(
    pydantic_object=UserInformation
)


fixing_parser = OutputFixingParser.from_llm(
    parser=parser,
    llm=model,
    max_retries=2,
)


prompt = ChatPromptTemplate.from_template(
    EXTRACT_USER_INFO_PROMPT
)


extract_user_info_chain = prompt | model | fixing_parser


def _normalize_existing_user_info(
    existing_user_info: UserInformation | dict | str | None,
) -> str:
    """
    Convert existing user information into a predictable JSON string
    before sending it to the LLM.
    """

    if existing_user_info is None:
        return "{}"

    if isinstance(existing_user_info, UserInformation):
        return existing_user_info.model_dump_json(indent=2)

    if isinstance(existing_user_info, dict):
        return json.dumps(
            existing_user_info,
            ensure_ascii=False,
            indent=2,
        )

    if isinstance(existing_user_info, str):
        existing_user_info = existing_user_info.strip()

        if not existing_user_info:
            return "{}"

        # If the caller already supplied JSON, preserve it.
        try:
            parsed = json.loads(existing_user_info)

            return json.dumps(
                parsed,
                ensure_ascii=False,
                indent=2,
            )

        except json.JSONDecodeError:
            # Keep backward compatibility if somebody passes
            # plain text instead of JSON.
            return existing_user_info

    raise TypeError(
        "existing_user_info must be UserInformation, dict, str, or None"
    )


def extract_user_information(
    resume_content: str,
    existing_user_info: UserInformation | dict | str | None = None,
    user_instruction: str = "",
) -> UserInformation:

    if not resume_content or not resume_content.strip():
        raise ValueError("resume_content must not be empty")

    existing_user_info_content = _normalize_existing_user_info(
        existing_user_info
    )

    result = extract_user_info_chain.invoke(
        {
            "resume_content": resume_content.strip(),

            "existing_user_info": existing_user_info_content,

            "user_instruction": (
                user_instruction.strip()
                if user_instruction
                else ""
            ),

            "format_instructions": parser.get_format_instructions(),
        }
    )

    return result

In [5]:
resume = """
# **Anirban Das** 

**Adm. No.** 23JE0104 � 6290375587 � My <u>portfolio website-https://anirban-das-portfolio.vercel.app/</u>
� dasanirban268@gmail.com � <u>linkedin.com/in/anirbandas</u> � <u>github.com/anirban2005143a</u> 

## Education 

### **Indian Institute of Technology (Indian School of Mines), Dhanbad** 

_Bachelor of Technology in Computer Science and Engineering (GPA: 8.29 / 10.00)_ 

Expected May 2027 _Dhanbad, Jharkhand_ 

- **Relevant Coursework:** Data Structures and Algorithms (C++), Database Management System, Compiler Design, Computer Organization , Computer Architecture, Operating Systems. 

## Experience 

### **Google** _|_ **_Software Engineer Intern_** 

May 2026 – July 2026 

- Engineered a new linting rule and automated batch validation pipeline in Java and TypeScript to detect structural errors across cloud contract templates. 

- Enhanced core template management interfaces by developing a responsive document comparison dialog box, a read/write mode selector for gDoc, and an embedded AI assistant chat widget. 

- Contributed 7,350+ lines of production code across 22 peer-reviewed changelists and created analytical dashboards for contract monitoring and template health tracking. 

## Projects 

**<u>Code Fusion</u>** _| React.js, Flask, Express.js, MongoDB, Tailwind CSS |_ _<u>GitHub</u> | Deployed Project_ 

- An online code editor supporting real-time collaboration, multiple languages, and customizable themes. 

- Enables multiple developers to collaborate in real-time with live cursor tracking and integrated in-app chat for seamless communication. 

- Integrates auto-completion, real-time syntax error detection, and persistent code-saving functionality, while offering various themes, multi-language support, and a collapsible sidebar for efficient file management. 

**<u>JobPilot</u>** _| Python (FastAPI), TypeScript (Next.js), Langchain, WebSockets |_ _<u>GitHub</u> |_ _<u>Video</u>_ 

- Developed a full-stack automation system using **FastAPI** , **Next.js** , and **HuggingFace LLMs** to automate resume extraction and job matching, reducing manual application time by **80%** . 

- Architected a scalable background worker system with **asynchronous processing** , **WebSockets** for real-time updates, and **FileLock** synchronization to manage concurrent multi-user job lifecycles. 

- Built an **LLM-powered** pipeline using **prompt engineering** for job ranking and categorization, featuring a manual review workflow and a local mock portal for safe system validation. 

**NoteBridge** _| React.js, Express.js, MongoDB, Bootstrap |_ _<u>GitHub</u> | Deployed Project_ 

- A **feature-rich note-taking and sharing platform** that enables **structured organization** through folders and facilitates **controlled file sharing** . 

- Enables **interactive engagement** through features like **likes** , **comments** , and **shares** . 

- Provides a **comprehensive profile page** displaying total posts, followers, following, and an **organized archive of past posts** for easy access and engagement. 

## Technical Skills 

**AI/ML & Agents** : LangChain, LangGraph, TensorFlow, Keras, Deep Learning, ANN, CNN, LSTM. **Technologies** : Node.js, FastAPI, Express.js, Docker, Next.js, React.js, Tailwind CSS, Three.js, GSAP. **Database & Cloud** : MongoDB, PostgreSQL, Vector Databases. 

## Achievements 

- Secured **4th** rank at **HaXplore** _|_ **CodeFest’25** , organized by **IIT BHU!** 

- **Winner** - of Winter Of Code 6.O (in Web Development Division) a one-month long hackathon conducted by **CyberLabs** , IIT(ISM) Dhanbad. _| Deployed Project_ 

## Social Engagements 

- Member of CyberLabs -Tech society of IIT ISM Dhanbad 

- Member of Aquatics Team - Swimming Team of IIT ISM Dhanbad. 

- Represented IIT Dhanbad at the 37th INTER IIT AQUATICS MEET 2023 held at IIT Gandhinagar and secured **4th place** in 200m Individual Medley . 


"""

In [6]:
existing_user_info = UserInformation(
    full_name="Anirban Das",
    phone="9876543210",
    email="anirban@gmail.com",
    linkedin_url="linkedin.com/in/anirban",
    github_url="github.com/anirban",
    portfolio_url="https://anirban.dev",
)

In [8]:
result = extract_user_information(
    resume_content=resume,
    existing_user_info=existing_user_info,
)

print(result.model_dump())

{'full_name': 'Anirban Das', 'phone': '9876543210, 6290375587', 'linkedin_url': 'https://linkedin.com/in/anirbandas', 'github_url': 'https://github.com/anirban2005143a', 'portfolio_url': 'https://anirban-das-portfolio.vercel.app/', 'email': 'anirban@gmail.com, dasanirban268@gmail.com'}
